In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DataType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name='ecommerce'

## Brands

In [0]:
brand_schema = StructType([
    StructField('brand_code',StringType(), False),
    StructField('brand_name',StringType(),True),
    StructField('category_code',StringType(),True)
])

row_data_path= "/Volumes/ecommerce/source_data/row/brands/*.csv"

df_brands = spark.read.option("heade","true").option("delimeter",",").schema(brand_schema).csv(row_data_path)
df_brands = df_brands.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

df_brands.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

## Category

In [0]:
category_schema = StructType([
    StructField('category_code',StringType(),False),
    StructField('category_name',StringType(),True)
])

row_data_path="/Volumes/ecommerce/source_data/row/category/*.csv"
df_category = spark.read.option('header',"true").option("delimeter",",").schema(category_schema).csv(row_data_path)
df_category = df_category.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())


df_category.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_category")

## Customers

In [0]:
customers_schema = StructType([
    StructField('customer_id', StringType(),False),
    StructField('phone',StringType(),True),
    StructField('country_code',StringType(),True),
    StructField('country',StringType(),True),
    StructField('state',StringType(),True)
])

row_data_path="/Volumes/ecommerce/source_data/row/customers/*.csv"
df_customers = spark.read.option('header',"true").option("delimeter",",").schema(customers_schema).csv(row_data_path)
df_customers = df_customers.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())    

df_customers=df_customers.withColumn(
    "phone",
    F.col("phone").cast("double")
    )


df_customers.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")

## date

In [0]:
date_schema = StructType([
    StructField('date',StringType(),True),
    StructField('year',IntegerType(),True),
    StructField('day_name', StringType(),True),
    StructField('quarter',IntegerType(),True),
    StructField('week_of_year',IntegerType(),True)
])


row_data_path="/Volumes/ecommerce/source_data/row/date/*.csv"
df_date = spark.read.option('header',"true").option("delimeter",",").schema(date_schema).csv(row_data_path)
df_date = df_date.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

df_date.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_date")

## Products

In [0]:
product_schema = StructType([
    StructField('product_id',StringType(),False),
    StructField('sku',StringType(),True),
    StructField('category_code',StringType(),True),
    StructField('brand_code',StringType(),True),
    StructField('color',StringType(),True),
    StructField('size',StringType(),True),
    StructField('material',StringType(),True),
    StructField('weight_grams',StringType(),True),
    StructField('length_cm',StringType(),True),
    StructField('width_cm',FloatType(),True),
    StructField('height_cm',FloatType(),True),
    StructField('rating_count',IntegerType(),True)
])

row_data_path=f"/Volumes/ecommerce/source_data/row/products/*.csv"
df_product = spark.read.option('header',"true").option("delimeter",",").schema(product_schema).csv(row_data_path)
df_product = df_product.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

df_product=df_product.withColumn(
    "product_id",
    F.col("product_id").cast("bigint")
    )

In [0]:
df_product.display()

In [0]:
df_product.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_product")